Wifi data processing
For given wifi data ('updated_densitymap_hourly_cleaned.csv'), process and make some filter actions.

3.1  Exclude building type include 'Residential', 'Parking', 'Housing' 'Commercial'.

3.2  For data recorded in same building, same floor, get the average and max density cnt( density cnt refers to the number of people in the space at a given point in time), and get average and max density ratio.


In [1]:
import pandas as pd

# READ CSV FILE
df = pd.read_csv('updated_densitymap_hourly_cleaned.csv')
df['building_type'] = df['building_type'].str.strip()

# exclude 'building_type' 列中为 'Residential', 'Parking', 'Housing' 'Commercial' 
filtered_df = df[~df['building_type'].isin(['Residential', 'Parking', 'Housing', 'Commercial'])]


# calculate max and mean density cnt based on same building floor and building desc
# cnt refers to the number of people in the space at a given point in time
grouped_values = filtered_df.groupby(['building_floor', 'building_desc'], as_index=False).agg(
    mean_density_cnt=('density_cnt', 'mean'),
    max_density_cnt=('density_cnt', 'max')
)

#only record first column for each same building floor + building description
first_row_flags = filtered_df.groupby(['building_floor', 'building_desc']).cumcount() == 0
representative_rows = filtered_df[first_row_flags]

# add mean and max density into file
final_representative_rows = pd.merge(representative_rows, grouped_values, on=['building_floor', 'building_desc'], how='left')

# delete 'date'  'hour' 'density_cnt''dc_ratio'
final_representative_rows_cleaned = final_representative_rows.drop(columns=['date', 'hour','density_cnt','dc_ratio'], errors='ignore')

# add  'average_dc_ratio' and max dc ratio
final_representative_rows_cleaned['average_dc_ratio'] = final_representative_rows_cleaned['mean_density_cnt'] / final_representative_rows_cleaned['capacity']
 
final_representative_rows_cleaned['max_dc_ratio'] = final_representative_rows_cleaned['max_density_cnt'] / final_representative_rows_cleaned['capacity']

# create processed file
final_representative_rows_cleaned.to_csv('wifi_update.csv', index=False)

final_representative_rows_cleaned.head()

,building_floor,building_desc,latitude,longitude,building_type,capacity,mean_density_cnt,max_density_cnt,average_dc_ratio,max_dc_ratio
0,1,771 COMMONWEALTH AVENUE,42.3510,-71.1080,Student Support,73,25.448998,309,0.348616,4.232877
1,2,771 COMMONWEALTH AVENUE,42.3510,-71.1080,Student Support,92,30.477226,283,0.331274,3.076087
2,1,775 COMMONWEALTH AVENUE,42.3509,-71.1090,Student Support,534,78.563533,534,0.147123,1.000000
3,1,1 UNIVERSITY ROAD,42.3512,-71.1100,Academic,307,51.585595,351,0.168031,1.143322
4,1,745 - 755 COMMONWEALTH AVENUE,42.3505,-71.1073,Academic,129,40.219413,319,0.311778,2.472868
